# GPT OSS 120B — RLVR Math Training (No-Tool)

Train `openai/gpt-oss-120b` via **Reinforcement Learning with Verifiable Rewards (RLVR)** using the Tinker `tinker_cookbook.rl` framework.

**Key Details:**
- Uses `tinker_cookbook.rl.train.main()` for the full GRPO training loop
- Custom `ProblemEnv` subclass for math answer verification via `\boxed{}`
- Importance sampling loss (GRPO-style advantage estimation)
- `GptOssRenderer` (Harmony format) for prompt/response rendering
- LoRA rank=32, group_size=4, batch_size=8
- Dataset: Competition math problems with integer/expression answers

In [ ]:
# ============================================================
# Cell 1: Configuration
# ============================================================

class RLConfig:
    model_name = "openai/gpt-oss-120b"
    lora_rank = 32
    learning_rate = 1e-5          # Lower LR for RL (vs 2e-4 for SFT)
    max_tokens = 24576            # Generation budget per rollout
    batch_size = 8                # Problems per training step
    group_size = 4                # Completions per problem (GRPO)
    max_steps = 100               # Total training iterations
    temperature = 1.0             # Sampling temperature for rollouts
    loss_fn = "importance_sampling"  # Standard for GRPO
    eval_every = 10               # Evaluate every N steps
    save_every = 25               # Checkpoint every N steps
    format_coef = 0.1             # Weight for format reward component
    log_path = "/kaggle/working/rl_logs"
    
    # Dataset path on Kaggle
    dataset_path = "/kaggle/input/aimo-rlvr-dataset/my_dataset.csv"
    
    # Optional: warm-start from SFT checkpoint
    # Set to None to train from scratch base model
    load_checkpoint_path = None  # e.g. "tinker://SESSION_ID:train:0/sampler_weights/NAME"

print("RLVR Config:")
for k, v in vars(RLConfig).items():
    if not k.startswith('_'):
        print(f"  {k}: {v}")

In [ ]:
# ============================================================
# Cell 2: Install Dependencies
# ============================================================

!pip install -q tinker tinker-cookbook

In [ ]:
# ============================================================
# Cell 3: Imports & API Key
# ============================================================

import os, re, math, logging, asyncio
from functools import partial
from collections.abc import Sequence
from dataclasses import dataclass
from typing import Literal, cast

import pandas as pd
import chz
import tinker

from tinker_cookbook import renderers, model_info
from tinker_cookbook.tokenizer_utils import get_tokenizer
from tinker_cookbook.rl.problem_env import ProblemEnv, ProblemGroupBuilder
from tinker_cookbook.rl.types import RLDataset, RLDatasetBuilder, EnvGroupBuilder
from tinker_cookbook.rl import train

# Try to import the cookbook's math grading utilities
try:
    from tinker_cookbook.recipes.math_rl.math_grading import (
        extract_boxed,
        grade_answer,
        run_with_timeout_signal,
    )
    HAS_MATH_GRADING = True
    print("\u2713 Loaded tinker_cookbook math grading (SymPy-based)")
except ImportError:
    HAS_MATH_GRADING = False
    print("\u26a0 Math grading not available, using string matching fallback")

# ---- SET YOUR TINKER API KEY HERE ----
os.environ["TINKER_API_KEY"] = "YOUR_API_KEY_HERE"  # <-- REPLACE THIS!

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("gpt-oss-rlvr")

print(f"Tinker SDK version: {tinker.__version__}")
print(f"API key set: {'TINKER_API_KEY' in os.environ and os.environ['TINKER_API_KEY'] != 'YOUR_API_KEY_HERE'}")

In [ ]:
# ============================================================
# Cell 4: Load & Inspect Dataset
# ============================================================

df = pd.read_csv(RLConfig.dataset_path)

print(f"Dataset loaded: {len(df)} problems")
print(f"Columns: {list(df.columns)}")
print(f"\nAnswer types:")
print(f"  Numeric (int-like): {df['answer'].apply(lambda x: str(x).lstrip('-').isdigit()).sum()}")
print(f"  Other:             {(~df['answer'].apply(lambda x: str(x).lstrip('-').isdigit())).sum()}")

# Preview
print(f"\n{'='*60}")
for i in range(min(3, len(df))):
    row = df.iloc[i]
    print(f"\n[Problem {row['id']}]")
    print(f"  Q: {str(row['problem'])[:150]}...")
    print(f"  A: {row['answer']}")

In [ ]:
# ============================================================
# Cell 5: Custom Math Environment (ProblemEnv subclass)
# ============================================================

def extract_boxed_answer(text: str) -> str | None:
    """Extract content from the last \\boxed{...} in text, handling nested braces."""
    key = r"\boxed{"
    idx = text.rfind(key)
    if idx < 0:
        return None
    i = idx + len(key)
    depth = 1
    while i < len(text) and depth:
        if text[i] == "{": depth += 1
        elif text[i] == "}": depth -= 1
        i += 1
    return text[idx + len(key):i - 1].strip() if depth == 0 else None


def safe_grade_answer(given: str, ground_truth: str, timeout: float = 2.0) -> bool:
    """Grade an answer using SymPy if available, else string matching."""
    if HAS_MATH_GRADING:
        try:
            result = run_with_timeout_signal(
                grade_answer,
                args=(given, ground_truth),
                timeout_seconds=int(math.ceil(timeout)),
            )
            if result is not None:
                return result
        except Exception:
            pass
    
    # Fallback: normalize and compare strings
    def normalize(s: str) -> str:
        s = s.strip()
        s = s.replace(" ", "")
        s = s.replace(",", "")
        # Remove trailing .0 for integer comparison
        if s.endswith(".0"):
            s = s[:-2]
        return s.lower()
    
    return normalize(given) == normalize(ground_truth)


class AIMOMathEnv(ProblemEnv):
    """Single-turn math environment for AIMO competition problems.
    
    Rewards:
      - correct_answer: +1.0 if \\boxed{} matches ground truth
      - format: +0.1 bonus if \\boxed{} present (even if wrong)
      - Total = format_coef * (format - 1) + correct_answer
        = -0.1 if no \\boxed{}, 0.0 if wrong boxed, 1.0 if correct
    """
    
    def __init__(
        self,
        problem: str,
        answer: str,
        renderer: renderers.Renderer,
        convo_prefix: list[renderers.Message] | None = None,
        format_coef: float = 0.1,
    ):
        super().__init__(renderer, convo_prefix, format_coef=format_coef)
        self.problem = problem
        self.answer = str(answer).strip()
    
    @classmethod
    def question_suffix(cls) -> str:
        return (
            "\n\nPlease reason step by step, and put your final answer "
            "within \\boxed{}."
        )
    
    def get_question(self) -> str:
        return self.problem + self.question_suffix()
    
    def check_format(self, sample_str: str) -> bool:
        """Check if the response contains a \\boxed{} answer."""
        return extract_boxed_answer(sample_str) is not None
    
    def check_answer(self, sample_str: str) -> bool:
        """Check if the \\boxed{} answer matches ground truth."""
        predicted = extract_boxed_answer(sample_str)
        if predicted is None:
            return False
        return safe_grade_answer(predicted, self.answer)
    
    def get_reference_answer(self) -> str:
        return self.answer


# Quick test
print("Testing AIMOMathEnv logic:")
print(f"  extract_boxed_answer('The answer is \\\\boxed{{42}}'): {extract_boxed_answer('The answer is \\boxed{42}')}")
print(f"  extract_boxed_answer('No boxed here'): {extract_boxed_answer('No boxed here')}")
print(f"  safe_grade_answer('42', '42'): {safe_grade_answer('42', '42')}")
print(f"  safe_grade_answer('43', '42'): {safe_grade_answer('43', '42')}")
print("\u2713 Environment logic OK")

In [ ]:
# ============================================================
# Cell 6: Custom RLDataset & RLDatasetBuilder
# ============================================================

class AIMODataset(RLDataset):
    """RL dataset wrapping our CSV of competition math problems."""
    
    def __init__(
        self,
        problems: list[dict],
        batch_size: int,
        group_size: int,
        renderer: renderers.Renderer,
        convo_prefix: list[renderers.Message] | None = None,
        format_coef: float = 0.1,
    ):
        self.problems = problems
        self.batch_size = batch_size
        self.group_size = group_size
        self.renderer = renderer
        self.convo_prefix = convo_prefix
        self.format_coef = format_coef
    
    def get_batch(self, index: int) -> Sequence[EnvGroupBuilder]:
        batch_start = (index * self.batch_size) % len(self.problems)
        batch_end = min(batch_start + self.batch_size, len(self.problems))
        
        # Wrap around if we've exhausted the dataset
        indices = list(range(batch_start, batch_end))
        if len(indices) < self.batch_size:
            remaining = self.batch_size - len(indices)
            indices.extend(range(0, remaining))
        
        builders = []
        for i in indices:
            p = self.problems[i]
            builder = ProblemGroupBuilder(
                env_thunk=partial(
                    AIMOMathEnv,
                    problem=p["problem"],
                    answer=str(p["answer"]),
                    renderer=self.renderer,
                    convo_prefix=self.convo_prefix,
                    format_coef=self.format_coef,
                ),
                num_envs=self.group_size,
                dataset_name="aimo",
            )
            builders.append(builder)
        return builders
    
    def __len__(self) -> int:
        # Allow cycling through the dataset for max_steps iterations
        # Return a large number so train loop doesn't stop early
        return max(1000, math.ceil(len(self.problems) / self.batch_size))


@chz.chz
class AIMODatasetBuilder(RLDatasetBuilder):
    """Builds train (and optionally test) datasets from our CSV."""
    
    dataset_path: str
    batch_size: int
    group_size: int
    model_name_for_tokenizer: str
    renderer_name: str
    format_coef: float = 0.1
    eval_split: int = 5  # Hold out N problems for eval
    seed: int = 42
    
    async def __call__(self) -> tuple[AIMODataset, AIMODataset | None]:
        import random
        
        # Load CSV
        df = pd.read_csv(self.dataset_path)
        all_problems = df.to_dict('records')
        
        # Shuffle deterministically
        rng = random.Random(self.seed)
        rng.shuffle(all_problems)
        
        # Train/eval split
        eval_problems = all_problems[:self.eval_split]
        train_problems = all_problems[self.eval_split:]
        
        logger.info(f"Dataset: {len(train_problems)} train, {len(eval_problems)} eval")
        
        # Create renderer
        tokenizer = get_tokenizer(self.model_name_for_tokenizer)
        renderer = renderers.get_renderer(self.renderer_name, tokenizer=tokenizer)
        
        train_dataset = AIMODataset(
            problems=train_problems,
            batch_size=self.batch_size,
            group_size=self.group_size,
            renderer=renderer,
            format_coef=self.format_coef,
        )
        
        eval_dataset = AIMODataset(
            problems=eval_problems,
            batch_size=len(eval_problems),  # Evaluate all at once
            group_size=1,  # Single completion for eval
            renderer=renderer,
            format_coef=self.format_coef,
        )
        
        return train_dataset, eval_dataset


print(f"\u2713 AIMODataset and AIMODatasetBuilder defined")
print(f"  With {len(df)} problems: {len(df) - 5} train + 5 eval")
print(f"  Each step: {RLConfig.batch_size} problems x {RLConfig.group_size} completions = {RLConfig.batch_size * RLConfig.group_size} rollouts")

In [ ]:
# ============================================================
# Cell 7: Verify Environment & Renderer
# ============================================================

# Quick integration check: make sure our env works with the renderer
from tinker_cookbook.tokenizer_utils import get_tokenizer as _get_tok

renderer_name = model_info.get_recommended_renderer_name(RLConfig.model_name)
print(f"Recommended renderer for {RLConfig.model_name}: {renderer_name}")

_tokenizer = _get_tok(RLConfig.model_name)
_renderer = renderers.get_renderer(renderer_name, tokenizer=_tokenizer)

# Create a test environment
test_row = df.iloc[0]
test_env = AIMOMathEnv(
    problem=test_row['problem'],
    answer=str(test_row['answer']),
    renderer=_renderer,
)

print(f"\nTest environment:")
print(f"  Question (first 200 chars): {test_env.get_question()[:200]}...")
print(f"  Reference answer: {test_env.get_reference_answer()}")

# Test prompt building
import asyncio

async def _test_prompt():
    obs, stop_cond = await test_env.initial_observation()
    return obs, stop_cond

obs, stop = asyncio.get_event_loop().run_until_complete(_test_prompt())
print(f"\nPrompt ModelInput:")
print(f"  Token count: {obs.length}")
print(f"  Stop sequences configured: {len(stop) if isinstance(stop, list) else 'yes'}")

# Decode and show first few tokens
all_tokens = []
for chunk in obs.chunks:
    all_tokens.extend(chunk.tokens)
decoded_preview = _tokenizer.decode(all_tokens[:100])
print(f"  Decoded preview (first 100 tokens): {decoded_preview[:300]}...")

print(f"\n\u2713 Environment + Renderer integration verified!")

In [ ]:
# ============================================================
# Cell 8: Configure & Launch RL Training
# ============================================================

# Build the dataset builder
renderer_name = model_info.get_recommended_renderer_name(RLConfig.model_name)

dataset_builder = AIMODatasetBuilder(
    dataset_path=RLConfig.dataset_path,
    batch_size=RLConfig.batch_size,
    group_size=RLConfig.group_size,
    model_name_for_tokenizer=RLConfig.model_name,
    renderer_name=renderer_name,
    format_coef=RLConfig.format_coef,
    eval_split=5,
    seed=42,
)

# Build the training config
config = train.Config(
    # Core
    model_name=RLConfig.model_name,
    learning_rate=RLConfig.learning_rate,
    dataset_builder=dataset_builder,
    max_tokens=RLConfig.max_tokens,
    log_path=RLConfig.log_path,
    
    # LoRA
    lora_rank=RLConfig.lora_rank,
    
    # Loss
    loss_fn=RLConfig.loss_fn,
    
    # Sampling
    temperature=RLConfig.temperature,
    
    # Evaluation & checkpointing
    eval_every=RLConfig.eval_every,
    save_every=RLConfig.save_every,
    max_steps=RLConfig.max_steps,
    
    # Renderer
    renderer_name=renderer_name,
    
    # Optional: warm-start from SFT
    load_checkpoint_path=RLConfig.load_checkpoint_path,
    
    # Logging
    num_groups_to_log=2,
)

print("\u2554" + "\u2550"*60 + "\u2557")
print("\u2551  GPT OSS 120B RLVR Training")
print("\u2560" + "\u2550"*60 + "\u2563")
print(f"\u2551  Model:          {config.model_name}")
print(f"\u2551  LoRA rank:      {config.lora_rank}")
print(f"\u2551  Learning rate:  {config.learning_rate:.1e}")
print(f"\u2551  Max tokens:     {config.max_tokens}")
print(f"\u2551  Batch size:     {RLConfig.batch_size} problems")
print(f"\u2551  Group size:     {RLConfig.group_size} completions/problem")
print(f"\u2551  Rollouts/step:  {RLConfig.batch_size * RLConfig.group_size}")
print(f"\u2551  Max steps:      {config.max_steps}")
print(f"\u2551  Loss fn:        {config.loss_fn}")
print(f"\u2551  Temperature:    {config.temperature}")
print(f"\u2551  Renderer:       {config.renderer_name}")
print(f"\u2551  Eval every:     {config.eval_every} steps")
print(f"\u2551  Save every:     {config.save_every} steps")
print(f"\u2551  Checkpoint:     {config.load_checkpoint_path or 'None (base model)'}")
print("\u255a" + "\u2550"*60 + "\u255d")
print("\n\u23f3 Launching RLVR training...\n")

# Run training
asyncio.run(train.main(config))

In [ ]:
# ============================================================
# Cell 9: Download Trained Weights
# ============================================================

import requests
import json
from pathlib import Path

def find_final_checkpoint(log_path: str) -> str | None:
    """Find the final checkpoint path from the training logs."""
    checkpoints_file = Path(log_path) / "checkpoints.jsonl"
    if not checkpoints_file.exists():
        print(f"No checkpoints file found at {checkpoints_file}")
        return None
    
    last_checkpoint = None
    with open(checkpoints_file) as f:
        for line in f:
            try:
                data = json.loads(line.strip())
                if 'sampler_path' in data:
                    last_checkpoint = data['sampler_path']
            except json.JSONDecodeError:
                continue
    return last_checkpoint


def download_weights(sampler_path: str, output_dir: str = "gpt_oss_120b_rlvr_weights"):
    """Download the trained LoRA weights from Tinker."""
    os.makedirs(output_dir, exist_ok=True)
    
    service_client = tinker.ServiceClient()
    rest_client = service_client.create_rest_client()
    url_resp = rest_client.get_checkpoint_archive_url_from_tinker_path(sampler_path).result()
    
    print(f"Downloading checkpoint from: {sampler_path}")
    r = requests.get(url_resp.url, stream=True)
    r.raise_for_status()
    total_bytes = int(r.headers.get('content-length', 0))
    print(f"  File size: {total_bytes / 1e9:.2f} GB")
    
    output_file = os.path.join(output_dir, "lora_checkpoint.tar")
    downloaded = 0
    with open(output_file, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
            f.write(chunk)
            downloaded += len(chunk)
            if total_bytes:
                pct = 100 * downloaded / total_bytes
                print(f"\r  Progress: {pct:.1f}% ({downloaded/1e9:.2f}/{total_bytes/1e9:.2f} GB)", end="")
    
    print(f"\n  \u2713 Saved: {output_file} ({downloaded / 1e9:.2f} GB)")
    return output_file


# Find and download the final checkpoint
sampler_path = find_final_checkpoint(RLConfig.log_path)
if sampler_path:
    print(f"Found checkpoint: {sampler_path}")
    download_weights(sampler_path)
else:
    print("No checkpoint found. Training may not have completed.")

In [ ]:
# ============================================================
# Cell 10: Quick Evaluation (Optional)
# ============================================================
# Run a few test inferences with the trained model

async def quick_eval(sampler_path: str, test_problems: list[dict], n: int = 5):
    """Quick evaluation: sample from trained model and check answers."""
    service_client = tinker.ServiceClient()
    sampling_client = service_client.create_sampling_client(model_path=sampler_path)
    
    tokenizer = get_tokenizer(RLConfig.model_name)
    renderer = renderers.get_renderer(
        model_info.get_recommended_renderer_name(RLConfig.model_name),
        tokenizer=tokenizer,
    )
    
    correct = 0
    total = min(n, len(test_problems))
    
    for i in range(total):
        p = test_problems[i]
        question = p['problem'] + AIMOMathEnv.question_suffix()
        messages = [{"role": "user", "content": question}]
        
        prompt = renderer.build_generation_prompt(messages)
        stop_sequences = renderer.get_stop_sequences()
        
        params = tinker.SamplingParams(
            max_tokens=RLConfig.max_tokens,
            temperature=0.6,  # Lower temp for eval
            stop=stop_sequences,
        )
        
        output = await sampling_client.sample_async(
            prompt, sampling_params=params, num_samples=1
        )
        
        response_msg, success = renderer.parse_response(output.sequences[0].tokens)
        content = renderers.get_text_content(response_msg)
        
        predicted = extract_boxed_answer(content)
        gold = str(p['answer']).strip()
        is_correct = predicted is not None and safe_grade_answer(predicted, gold)
        
        if is_correct:
            correct += 1
        
        print(f"[{i+1}/{total}] {'\u2713' if is_correct else '\u2717'}")
        print(f"  Q: {p['problem'][:100]}...")
        print(f"  Predicted: {predicted}")
        print(f"  Gold:      {gold}")
        print()
    
    acc = 100 * correct / total if total > 0 else 0
    print(f"{'='*40}")
    print(f"  Accuracy: {correct}/{total} = {acc:.1f}%")
    print(f"{'='*40}")


# Run quick eval
if sampler_path:
    test_problems = df.to_dict('records')[:10]
    asyncio.run(quick_eval(sampler_path, test_problems))
else:
    print("Skipping evaluation - no checkpoint available")